# Aula 16 — Retrieval-Augmented Generation (RAG)

Agora vamos combinar capacidades estudadas separadamente:

```text
Aula 14 → generation
Aula 15 → retrieval + grounding

Aula 16 → RAG
```

Pergunta central:

> **Como combinar retrieval e generation sem perder rastreabilidade da evidência?**

## Objetivos

Ao final, você deverá conseguir:

- explicar a arquitetura mínima de RAG;
- distinguir generation-only de retrieval-augmented generation;
- observar retrieval, evidence pack e context construction separadamente;
- produzir uma grounded answer com attribution;
- analisar top-k sensitivity;
- reconhecer insufficient evidence;
- localizar retrieval, context, generation e attribution failures;
- discutir utility, custo, latência e risco.

## Glossário da aula

Conceitos centrais no **Glossário Vivo**:

**[RAG](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#retrieval-augmented-generation-rag) · [Generation-only](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#generation-only) · [Construção de contexto](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#construção-de-contexto) · [Resposta fundamentada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#resposta-fundamentada) · [Groundedness](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#groundedness) · [Atribuição de fonte](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#atribuição-de-fonte) · [Falha de retrieval](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falha-de-retrieval) · [Falha de contexto](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falha-de-contexto) · [Falha de geração](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falha-de-geração) · [Evidência insuficiente](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#evidência-insuficiente)**

A aula também reutiliza retrieval, evidence pack, grounding, generation, top-k retrieval, abstention e utility.


## 1. Por que RAG existe?

Um modelo generativo pode produzir texto sem consultar uma fonte externa.

Isso é útil, mas cria uma limitação:

```text
query
→ generator
→ answer
```

Se a informação necessária está fora do contexto, o sistema precisa de outra capacidade.

RAG adiciona retrieval antes da geração:

```text
query
→ retrieval
→ evidence
→ context
→ generator
→ grounded answer
```

O importante é observar cada etapa separadamente.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Ambiente da Aula 16 pronto.")
print("Internet OFF: todos os componentes são locais e determinísticos.")


## 2. Corpus didático

Vamos trabalhar com um pequeno corpus sobre políticas de atendimento.

Ele contém fatos específicos que nosso gerador didático **não conhece por padrão**.

Isso nos permite comparar generation-only com RAG sem depender de uma API externa.


In [ ]:
documents = pd.DataFrame([
    {"doc_id": "C1", "text": "O prazo para reembolso é de até cinco dias úteis."},
    {"doc_id": "C2", "text": "A senha pode ser redefinida pela página de login."},
    {"doc_id": "C3", "text": "A troca de endereço de e-mail deve ser feita nas configurações do perfil."},
    {"doc_id": "C4", "text": "A autenticação em dois fatores pode ser ativada na área de segurança da conta."},
    {"doc_id": "C5", "text": "Solicitações de cancelamento devem ser registradas antes da renovação do contrato."},
])

display(documents)


## 3. Baseline: generation-only

Antes de adicionar retrieval, vamos estabelecer um baseline arquitetural.

Nosso gerador didático se comporta de forma conservadora: sem evidência explícita, ele não inventa uma resposta específica.


In [ ]:
def generation_only(question):
    return {
        "answer": "Não tenho evidência suficiente para responder com segurança.",
        "source": None,
        "grounded": False,
        "mode": "generation-only",
    }

question = "Qual é o prazo para reembolso?"
baseline = generation_only(question)

display(pd.DataFrame([baseline]))


## 4. Retrieval

Reutilizamos o mecanismo da Aula 15.

A função abaixo transforma corpus + query em TF-IDF, calcula cosine similarity e devolve ranking + top-k.


In [ ]:
def retrieve(query, documents, top_k=2):
    vectorizer = TfidfVectorizer(lowercase=True)
    matrix = vectorizer.fit_transform(documents["text"].tolist() + [query])

    doc_matrix = matrix[:-1]
    query_vector = matrix[-1]
    scores = cosine_similarity(query_vector, doc_matrix).ravel()

    result = documents.copy()
    result["score"] = scores
    result = result.sort_values("score", ascending=False).reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))

    return result.head(top_k)

retrieved = retrieve(question, documents, top_k=2)
display(retrieved)


## 5. Evidence pack

O resultado do retrieval será transformado em uma estrutura explícita.

Isso cria uma fronteira clara:

```text
retrieval result
→ evidence pack
→ context construction
```


In [ ]:
def build_evidence_pack(query, retrieved_df):
    return {
        "query": query,
        "evidence": [
            {
                "id": row["doc_id"],
                "text": row["text"],
                "score": float(row["score"]),
            }
            for _, row in retrieved_df.iterrows()
        ],
    }

evidence_pack = build_evidence_pack(question, retrieved)
display(pd.DataFrame(evidence_pack["evidence"]))


## 6. Context construction

Agora organizamos instrução, evidências e pergunta em um contexto visível.

Nada fica escondido em um helper opaco.


In [ ]:
def build_context(evidence_pack):
    evidence_lines = "\n".join(
        f"[{item['id']}] {item['text']}"
        for item in evidence_pack["evidence"]
    )

    return (
        "INSTRUÇÃO\n"
        "Use apenas a evidência fornecida. Se ela for insuficiente, sinalize isso.\n\n"
        "EVIDÊNCIA\n"
        f"{evidence_lines}\n\n"
        "PERGUNTA\n"
        f"{evidence_pack['query']}"
    )

context = build_context(evidence_pack)
print(context)


## 7. Gerador didático determinístico

Ainda não usamos um LLM real.

O objetivo desta aula é entender a arquitetura RAG, não medir um fornecedor ou modelo.

O gerador abaixo:

1. procura evidência compatível com a pergunta;
2. responde apenas quando encontra suporte;
3. mantém attribution;
4. abstém quando a evidência é insuficiente.

Ele é deliberadamente simples e transparente.


In [ ]:
def didactic_generator(question, evidence_pack, min_score=0.05):
    evidence = evidence_pack["evidence"]

    if not evidence or evidence[0]["score"] < min_score:
        return {
            "answer": "Evidência insuficiente para responder.",
            "source": None,
            "grounded": False,
            "mode": "rag-didactic",
        }

    best = evidence[0]
    text = best["text"]

    if "prazo" in question.lower() and "reembolso" in question.lower() and "reembolso" in text.lower():
        match = re.search(r"até\s+cinco\s+dias\s+úteis", text.lower())
        if match:
            return {
                "answer": "O prazo para reembolso é de até cinco dias úteis.",
                "source": best["id"],
                "grounded": True,
                "mode": "rag-didactic",
            }

    return {
        "answer": "Evidência recuperada, mas insuficiente para uma resposta específica.",
        "source": None,
        "grounded": False,
        "mode": "rag-didactic",
    }

rag_answer = didactic_generator(question, evidence_pack)
display(pd.DataFrame([rag_answer]))


## 8. Generation-only vs RAG

Agora podemos comparar os dois fluxos.

A comparação não prova que RAG é universalmente melhor. Ela mostra que, **neste cenário**, retrieval adiciona acesso a uma informação externa específica.


In [ ]:
comparison = pd.DataFrame([
    {
        "system": "generation-only",
        "has_retrieval": False,
        "has_attribution": baseline["source"] is not None,
        "grounded": baseline["grounded"],
        "answer": baseline["answer"],
    },
    {
        "system": "rag-didactic",
        "has_retrieval": True,
        "has_attribution": rag_answer["source"] is not None,
        "grounded": rag_answer["grounded"],
        "answer": rag_answer["answer"],
    },
])

display(comparison)


## 9. Top-k sensitivity

Mais evidência não significa automaticamente melhor contexto.

Vamos observar quanto conteúdo entra no evidence pack quando top-k aumenta.


In [ ]:
rows = []

for k in [1, 2, 3, 5]:
    r = retrieve(question, documents, top_k=k)
    rows.append({
        "top_k": k,
        "evidence_items": len(r),
        "total_characters": int(r["text"].str.len().sum()),
        "best_score": float(r.iloc[0]["score"]),
        "lowest_included_score": float(r.iloc[-1]["score"]),
    })

topk_df = pd.DataFrame(rows)
display(topk_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(topk_df["top_k"], topk_df["total_characters"], marker="o")
ax.set(
    xlabel="top-k",
    ylabel="Caracteres no contexto",
    title="Crescimento do contexto com top-k"
)
ax.grid(alpha=.25)
plt.show()
plt.close(fig)


### Interpretação

Aumentar top-k pode:

- ampliar cobertura;
- incluir informação irrelevante;
- aumentar custo;
- aumentar latência;
- criar conflitos;
- tornar o contexto mais difícil de usar.

Portanto, top-k é uma decisão de sistema, não um parâmetro a maximizar.


## 10. Missing evidence e abstention

Agora vamos perguntar algo que **não existe** no corpus.

Esse caso é importante: um sistema grounded deve reconhecer quando não possui evidência suficiente.


In [ ]:
missing_question = "Qual é o telefone do suporte?"
missing_retrieved = retrieve(missing_question, documents, top_k=2)
missing_pack = build_evidence_pack(missing_question, missing_retrieved)
missing_answer = didactic_generator(missing_question, missing_pack, min_score=0.05)

display(missing_retrieved)
display(pd.DataFrame([missing_answer]))


## 11. Localizando falhas

RAG é um sistema composto. Dizer apenas “o RAG errou” é pouco informativo.

Vamos separar quatro classes:

```text
retrieval failure
context failure
generation failure
attribution failure
```


In [ ]:
failure_scenarios = pd.DataFrame([
    {
        "scenario": "A",
        "description": "O corpus contém a resposta, mas o trecho correto não aparece no top-k.",
        "failure": "retrieval failure",
    },
    {
        "scenario": "B",
        "description": "O trecho correto foi recuperado, mas foi removido na construção do contexto.",
        "failure": "context failure",
    },
    {
        "scenario": "C",
        "description": "O contexto informa cinco dias úteis, mas a resposta afirma dez dias.",
        "failure": "generation failure",
    },
    {
        "scenario": "D",
        "description": "A resposta está correta, mas não indica qual evidência a sustenta.",
        "failure": "attribution failure",
    },
])

display(failure_scenarios)


## 12. Utility e trade-offs

Adicionar RAG introduz nova capacidade, mas também nova complexidade.

Compare:

```text
generation-only
→ menor pipeline
→ menos retrieval cost
→ sem acesso explícito ao corpus

RAG
→ retrieval + ranking + contexto + geração
→ maior custo/latência
→ acesso explícito à evidência
→ maior rastreabilidade potencial
```

A pergunta correta é:

> **A necessidade de acesso e rastreabilidade de evidência justifica a complexidade adicional nesta tarefa?**


## 13. Exercício 1 — Construa o contexto

Use `evidence_pack` e escreva uma função ou expressão que produza um bloco contendo:

- instrução;
- IDs das evidências;
- textos;
- pergunta.


In [ ]:
# Sua resposta aqui


### Dica

Você pode reutilizar a estrutura de `build_context()` e iterar sobre `evidence_pack["evidence"]`.


In [ ]:
# Solução executável
exercise_context = build_context(evidence_pack)
print(exercise_context)


## 14. Exercício 2 — Compare top-k

Execute retrieval para `top_k=1` e `top_k=3`.

Compare:

- número de evidências;
- quantidade de texto no contexto;
- presença de itens irrelevantes.


In [ ]:
# Sua resposta aqui


### Dica

Use `retrieve(question, documents, top_k=...)` e compare as linhas retornadas.


In [ ]:
# Solução executável
top1 = retrieve(question, documents, top_k=1)
top3 = retrieve(question, documents, top_k=3)

print("Top-1")
display(top1)

print("Top-3")
display(top3)


## 15. Exercício 3 — O sistema deve responder?

Pergunta:

`"Qual é o telefone do suporte?"`

O corpus não contém telefone.

Escolha uma ação:

1. responder mesmo assim;
2. sinalizar evidência insuficiente;
3. ampliar a busca;
4. escalar.

Explique por que.


### Resposta de referência

O sistema não deve inventar um telefone.

Uma resposta adequada é sinalizar **evidência insuficiente** e, dependendo do produto, ampliar a busca ou escalar o caso.

Esse comportamento é uma forma de **abstention**.


## 16. Exercício 4 — Localize a falha

Associe:

- trecho correto não recuperado;
- trecho correto recuperado mas removido do contexto;
- contexto correto mas resposta errada;
- resposta correta sem fonte;

às quatro categorias de failure estudadas.


### Resposta de referência

- trecho correto não recuperado → **retrieval failure**;
- trecho correto removido do contexto → **context failure**;
- contexto correto + resposta errada → **generation failure**;
- resposta correta sem fonte → **attribution failure**.


## 17. Exercício 5 — RAG é necessário?

Compare:

1. classificar uma mensagem em três categorias fixas;
2. responder perguntas sobre políticas que mudam e ficam armazenadas em um corpus externo.

Em qual caso RAG acrescenta uma capacidade mais claramente necessária? Justifique sem declarar uma arquitetura universalmente superior.


### Resposta de referência

No primeiro caso, um classificador pode ser suficiente porque o problema possui espaço de saída fechado e não exige consulta externa.

No segundo, retrieval pode adicionar acesso explícito a informações externas e atualizáveis. RAG passa a fazer sentido quando precisamos combinar essa evidência com uma resposta em linguagem natural.

A escolha depende da tarefa, da qualidade exigida, do custo, da latência e do risco.


## 18. Reprodutibilidade

Esta versão:

- usa Internet OFF;
- usa GPU OFF;
- não chama API externa;
- não usa vector database;
- não usa framework de RAG;
- executa retrieval real com TF-IDF + cosine similarity;
- usa gerador didático determinístico;
- não apresenta o gerador como benchmark de LLM.

Quando incorporarmos um LLM real, a medição deverá seguir a arquitetura **AUTHOR / EVIDENCE → STUDENT**.


## 19. Síntese

RAG não é uma única operação.

```text
query
→ retrieval
→ evidence
→ context construction
→ generation
→ grounded answer
→ attribution
→ evaluation
```

Cada estágio pode falhar de maneira diferente.

O ganho de capacidade só justifica a arquitetura quando produz utility suficiente para compensar custo, latência, risco e complexidade.

### Próxima ponte

A próxima pergunta será:

> **Se o sistema já consegue recuperar evidência e gerar uma resposta grounded, como permitir que ele execute ações controladas sobre ferramentas externas?**

Essa pergunta abre o próximo bloco: **Tools and Workflows**.



---

## Continue no TIL

← **[Anterior: Aula 15 — Retrieval, Semantic Search and Grounding](https://www.kaggle.com/code/pedrogentil/til-15-retrieval-semantic-search-grounding)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **[Próxima: Tools and Workflows — em preparação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/ROADMAP.md)** →
